In [23]:
from dotenv import load_dotenv

In [24]:
import os

In [25]:
load_dotenv(override=True)

True

In [26]:
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

In [27]:
openai_api_key = os.getenv("API_TOKEN")

In [28]:
if openai_api_key:
    print(f"OpenAI API key exists and begins with: {openai_api_key[:10]}...")
else:
    print("OpenAI API key not found. Please set the API_TOKEN environment variable.")

OpenAI API key exists and begins with: sk-or-v1-a...


In [29]:
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [30]:
import json
from agents import Agent, Runner, function_tool, AsyncOpenAI, OpenAIChatCompletionsModel

In [31]:
@function_tool
def generate_sermon(concept: str) -> str:
    """
    Generate a sermon based on a spiritual concept like faith, peace, joy, love,
    forgiveness, hope, grace, patience, etc.
    
    Args:
        concept: The spiritual concept or theme for the sermon (e.g. 'faith', 'peace', 'joy')
    """
    return (
        f"Generate a sermon on the concept of '{concept}'. "
        f"Structure your response EXACTLY like this:\n\n"
        f"**SERMON TOPIC:** [A compelling sermon title about {concept}]\n\n"
        f"**KEY BIBLE VERSES:** [List 3-5 relevant Bible verses with full references]\n\n"
        f"**SERMON:**\n"
        f"[Write a clear, inspiring sermon of about 300-400 words on {concept}. "
        f"Reference the key verses naturally. Include introduction, body, and conclusion with a call to action.]"
    )

In [32]:
@function_tool
def find_verse(phrase: str) -> str:
    """
    Find the exact Bible reference (book, chapter, verse) for a phrase the user
    remembers from the Bible but doesn't know where it's located.
    
    Args:
        phrase: The phrase or partial quote the user remembers from the Bible
    """
    return (
        f"The user remembers this phrase from the Bible: '{phrase}'\n\n"
        f"Find the exact Bible reference. Respond with:\n"
        f"**REFERENCE:** [Book Chapter:Verse]\n"
        f"**FULL VERSE (KJV):** [The complete verse text]\n"
        f"**CONTEXT:** [1-2 sentences about the context]\n\n"
        f"If the phrase matches multiple verses, list the top 2-3 most likely matches. "
        f"If the phrase is not in the Bible, say so clearly."
    )

In [33]:
@function_tool
def bible_lookup(book: str = "", chapter: str = "", verse: str = "") -> str:
    """
    Look up the content of a specific Bible passage given book, chapter, and/or verse.
    The user must provide book, chapter, and verse for a specific lookup.
    
    Args:
        book: The book of the Bible (e.g. 'Genesis', 'John', 'Psalms')
        chapter: The chapter number (e.g. '3')
        verse: The verse number or range (e.g. '16' or '16-18')
    """
    has_book = bool(book and book.strip())
    has_chapter = bool(chapter and chapter.strip())
    has_verse = bool(verse and verse.strip())
    
    if has_book and has_chapter and has_verse:
        return (
            f"Look up {book} {chapter}:{verse} in the Bible.\n\n"
            f"Respond with:\n"
            f"**REFERENCE:** {book} {chapter}:{verse}\n"
            f"**TEXT (KJV):** [The full verse text]\n"
            f"**CONTEXT:** [Brief explanation of the passage's meaning and context]"
        )
    
    missing = []
    if not has_book:
        missing.append("book (e.g. Genesis, John, Psalms)")
    if not has_chapter:
        missing.append("chapter number")
    if not has_verse:
        missing.append("verse number")
    
    provided = []
    if has_book:
        provided.append(f"Book: {book}")
    if has_chapter:
        provided.append(f"Chapter: {chapter}")
    if has_verse:
        provided.append(f"Verse: {verse}")
    
    provided_str = ", ".join(provided) if provided else "nothing yet"
    missing_str = ", ".join(missing)
    
    return (
        f"The user wants to look up a Bible verse but hasn't provided complete info.\n"
        f"They provided: {provided_str}\n"
        f"They're missing: {missing_str}\n\n"
        f"Politely ask them to provide the missing: {missing_str}. "
        f"Give an example like 'John 3:16' (book chapter:verse)."
    )

In [34]:
bible_agent = Agent(
    name="Bible Agent",
    instructions=(
        "You are a knowledgeable and warm Bible assistant. You help users with:\n"
        "1. Generating sermons on spiritual concepts (use generate_sermon tool)\n"
        "2. Finding Bible verses from partial phrases (use find_verse tool)\n"
        "3. Looking up specific Bible passages (use bible_lookup tool)\n\n"
        "RULES:\n"
        "- Always use the appropriate tool for the user's request\n"
        "- For sermon requests, detect the concept and call generate_sermon\n"
        "- For phrase searches like 'where does it say...' or 'what verse says...', use find_verse\n"
        "- For specific lookups like 'read John 3:16', use bible_lookup\n"
        "- If the user gives an incomplete reference (missing book/chapter/verse), "
        "  still call bible_lookup with what they gave - the tool handles missing info\n"
        "- Be respectful, encouraging, and spiritually uplifting\n"
        "- Use KJV as the default translation unless asked otherwise"
    ),
    model="gpt-4o-mini",
    tools=[generate_sermon, find_verse, bible_lookup]
)

In [36]:
result = await Runner.run(bible_agent, "Where in the Bible does it say 'I can do all things through Christ'?")
print(result.final_output)

**REFERENCE:** Philippians 4:13  
**FULL VERSE (KJV):** "I can do all things through Christ which strengtheneth me."  
**CONTEXT:** This verse, written by the Apostle Paul, expresses his confidence in the strength and support he receives from Jesus Christ. It is part of a larger message where Paul talks about being content in all circumstances, reliant on God's provision.


In [37]:
async def chat():
    print("=" * 60)
    print("  BIBLE AGENT")
    print("=" * 60)
    print("\nI can help you with:")
    print("  1. Generate a sermon (e.g. 'sermon on faith')")
    print("  2. Find a verse from a phrase (e.g. 'where does it say I can do all things')")
    print("  3. Look up a verse (e.g. 'read John 3:16')")
    print("\nType 'quit' to exit.\n")
    
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            print("\nGod bless you! Goodbye.")
            break
        
        result = await Runner.run(bible_agent, user_input)
        print(f"\nBible Agent: {result.final_output}\n")

await chat()

  BIBLE AGENT

I can help you with:
  1. Generate a sermon (e.g. 'sermon on faith')
  2. Find a verse from a phrase (e.g. 'where does it say I can do all things')
  3. Look up a verse (e.g. 'read John 3:16')

Type 'quit' to exit.


Bible Agent: It seems like your message was incomplete. Could you please provide more details or specify what you would like assistance with? Whether it's a Bible verse, a sermon idea, or a specific passage, I'm here to help!


Bible Agent: **SERMON TOPIC:** The Gift of Salvation: Embracing God's Grace

**KEY BIBLE VERSES:** 
- John 3:16 - "For God so loved the world, that he gave his only begotten Son, that whosoever believeth in him should not perish, but have everlasting life."
- Ephesians 2:8-9 - "For by grace are ye saved through faith; and that not of yourselves: it is the gift of God: Not of works, lest any man should boast."
- Romans 10:9 - "That if thou shalt confess with thy mouth the Lord Jesus, and shalt believe in thine heart that God hath raise